<div align="center">

# 🎙️ OmniVoice Studio Pro
### Giao diện Voice Cloning & TTS Tiếng Việt trên Google Colab
</div>

> ### 💡 HƯỚNG DẪN TRƯỚC KHI CHẠY
>
> **Bước 1 — Bật GPU T4:** vào menu **Runtime (Thời gian chạy) → Change runtime type (Thay đổi loại thời gian chạy) → T4 GPU → Save (Lưu)**.  
> **Bước 2 — Chạy công cụ:** vào menu **Runtime (Thời gian chạy) → Run all (Chạy tất cả)** (hoặc bấm `Ctrl + F9`).  
> **Bước 3 — Sử dụng ngay:** Toàn bộ tiến trình chuẩn bị môi trường và tải model sẽ hiển thị qua thanh tiến trình trực quan bên dưới. Khi sẵn sàng, giao diện Web UI sẽ tự động xuất hiện ngay bên trong Colab kèm đường link công khai (`gradio.live`).


In [ ]:
#@title 🎙️ OmniVoice Studio Pro  { display-mode: "form" }
# Colab App Mode: Giao diện sạch sẽ, ẩn toàn bộ log kỹ thuật và code.

import os, sys, time, shutil, subprocess, warnings, logging, traceback
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. TRIỆT TIÊU LOG RÁC & CẢNH BÁO
# -----------------------------------------------------------------------------
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TQDM_DISABLE"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
warnings.filterwarnings("ignore")

for logger_name in ["huggingface_hub", "transformers", "urllib3", "filelock", "pydub"]:
    lg = logging.getLogger(logger_name)
    lg.setLevel(logging.ERROR)
    lg.propagate = False
logging.getLogger().setLevel(logging.ERROR)

from IPython.display import display, HTML, clear_output

def update_status(pct, title, detail, is_error=False):
    clear_output(wait=True)
    color = "#ef4444" if is_error else "linear-gradient(90deg, #6366f1, #06b6d4)"
    badge_bg = "#fee2e2" if is_error else "#eef2ff"
    badge_color = "#b91c1c" if is_error else "#4f46e5"
    
    html_card = f"""
    <div style="max-width:980px;margin:16px auto;padding:26px 30px;border:1px solid #e2e8f0;border-radius:22px;background:#ffffff;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;box-shadow:0 12px 36px rgba(0,0,0,0.06)">
      <div style="display:flex;justify-content:space-between;align-items:center;">
        <div style="font-weight:900;font-size:21px;color:#1e293b;display:flex;align-items:center;gap:12px;">
          <span style="font-size:24px;">🎙️</span> OmniVoice Studio Pro
        </div>
        <div style="font-size:13px;font-weight:800;color:{badge_color};background:{badge_bg};padding:5px 14px;border-radius:999px;">
          {pct}%
        </div>
      </div>
      <div style="margin-top:10px;color:#334155;font-size:15px;font-weight:600;">{title}</div>
      <div style="height:9px;background:#f1f5f9;border-radius:999px;margin-top:16px;overflow:hidden">
        <div style="height:100%;width:{pct}%;background:{color};border-radius:999px;transition:width 0.4s ease"></div>
      </div>
      <div style="margin-top:12px;font-size:13px;color:#64748b;line-height:1.5;">{detail}</div>
    </div>
    """
    display(HTML(html_card))

def _quiet_run(cmd, check=True):
    env = os.environ.copy()
    return subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=check, env=env, text=True)

# -----------------------------------------------------------------------------
# 2. TIẾN TRÌNH KHỞI TẠO HỆ THỐNG
# -----------------------------------------------------------------------------
try:
    # Bước 1: Kiểm tra GPU (10%)
    update_status(10, "Đang kiểm tra môi trường phần cứng…", "Kiểm tra GPU T4")
    try:
        gpu_name = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True
        ).strip()
    except Exception:
        gpu_name = ""

    if not gpu_name:
        update_status(
            100,
            "Chưa bật GPU tăng tốc!",
            "Vào menu: <b>Runtime → Change runtime type → Chọn T4 GPU → Save</b>, sau đó bấm <b>Run all</b> lại.",
            is_error=True
        )
        raise SystemExit(1)

    # Bước 2: Chuẩn bị mã nguồn dự án (25%)
    update_status(25, f"GPU sẵn sàng: {gpu_name}", "Đang chuẩn bị mã nguồn OmniVoice Studio Pro…")
    repo_dir = Path("/content/OmniVoice-Studio-Pro")
    os.chdir("/content")
    if not repo_dir.exists():
        _quiet_run(["git", "clone", "-q", "https://github.com/vietlh93/OmniVoice-Studio-Pro.git", str(repo_dir)])
    else:
        _quiet_run(["git", "-C", str(repo_dir), "pull", "-q"])

    os.chdir(str(repo_dir))
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))

    # Bước 3: Cài đặt thư viện dependencies (45%)
    update_status(45, "Đang cài đặt thư viện dependencies…", "Gỡ xung đột torchvision và cài đặt các gói cần thiết (khoảng 1-2 phút)")
    _quiet_run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"], check=False)
    _quiet_run([
        sys.executable, "-m", "pip", "install", "-q",
        "--disable-pip-version-check", "--no-warn-conflicts",
        "-r", "general/requirements.txt"
    ])
    _quiet_run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-U", "huggingface_hub"])

    # Bước 4: Tải các Model AI (60% - 90%)
    from huggingface_hub import snapshot_download

    models_to_download = [
        ("Higgs Audio V2 Tokenizer", "eustlb/higgs-audio-v2-tokenizer", "OmniVoice/model_higgs_audio_v2_tokenizer_local", 60),
        ("ASR Chunkformer Tiếng Việt", "khanhld/chunkformer-ctc-large-vie", "OmniVoice/model_ASR_chunkformer_local", 75),
        ("OmniVoice Tiếng Việt (KhanhTTS)", "kjanh/KhanhTTS-OmniVoice", "OmniVoice/modelOmniLocal", 90),
    ]

    for model_name, repo_id, local_dir, pct in models_to_download:
        update_status(pct, f"Đang tải {model_name}…", f"Repo: {repo_id} → {local_dir}")
        snapshot_download(
            repo_id=repo_id,
            local_dir=local_dir,
            ignore_patterns=["*.git*"],
        )

    # Bước 5: Mở Web UI Gradio (100%)
    update_status(100, "Hệ thống sẵn sàng!", "Đang khởi động giao diện Web UI…")
    time.sleep(1)
    clear_output(wait=True)

    # Đóng phiên Gradio cũ nếu người dùng Run all lại
    try:
        import gradio as gr
        gr.close_all()
    except Exception:
        pass

    import app
    app.demo.launch(
        inline=True,
        share=True,
        quiet=True,
        show_error=False,
        height=1250,
    )

except SystemExit:
    pass
except Exception as e:
    update_status(
        100,
        "Đã xảy ra lỗi khi khởi chạy!",
        f"Chi tiết: {e}<br><br>Hãy chọn <b>Runtime → Restart session</b> rồi bấm <b>Run all</b> lại.",
        is_error=True
    )
